# Engines

> bioMONAI training engines


In [ ]:
#| default_exp engines

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Standard library
# =================================

from __future__ import annotations 

import inspect 
from abc import ABC, abstractmethod 
from dataclasses import dataclass, field 
from typing import Any, Callable, Dict, Mapping, Optional, Type
from pathlib import Path

# =================================
# Scientific / data
# =================================
import numpy as np
import pandas as pd

# =================================
# Visualization
# =================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =================================
# Imaging
# =================================
# from skimage import util

# =================================
# PyTorch
# =================================
import torch.optim as toptim
from torch.cuda import is_available as is_cuda_available
from torch.nn.init import kaiming_normal_

# =================================
# fastai
# =================================
# import fastai.losses
# import fastai.metrics
# import fastai.optimizer

from fastai.callback.core import Callback
from fastai.callback.all import *

from fastai.data.all import (
    DataLoaders, Path, trainable_params, delegates,
    hasattrs, List, L, Normalize
)

from fastai.optimizer import Adam, OptimWrapper, Optimizer

from fastai.vision.all import (
    Any, BypassNewMeta, CSVLogger, ClassificationInterpretation,
    DataBlock, DisplayedTransform, Learner, ShowGraphCallback,
    create_vision_model, create_timm_model, default_split,
    get_c, ifnone, minimum, model_meta, slide, steep, store_attr, valley
)

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *
from bioMONAI.datasets import download_medmnist

The engine module provides advanced functionalities for model training, including configurable training loops and evaluation functions tailored for bioinformatics applications. This module is significantly valuable when there is a need for specific workflows and pipelines that meet specific requirements. For this reason, the classes fastTrainer and visionTrainer have been created, providing tailored implementations inheriting from the Learner class. 


## Base Classes

### Backend Registry

In [ ]:
#| export

TrainerFactory = Callable[..., Any]

BACKEND_REGISTRY: Dict[str, Type["TrainerBackend"]] = {}
TRAINER_REGISTRY: Dict[str, Dict[str, TrainerFactory]] = {}



In [ ]:
#| export

def _validate_name(name: str, kind: str) -> str:
    """
    Validate and normalize a registry name.

    Parameters
    ----------
    name
        Name to validate.
    kind
        Type of registry entry, used in error messages.

    Returns
    -------
    str
        Stripped and lowercase name.

    Raises
    ------
    ValueError
        If ``name`` is not a non-empty string.
    """
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"{kind} name must be a non-empty string.")

    return name.strip().lower()


In [ ]:
#| export

def register_backend(
    name: str,
) -> Callable[[Type["TrainerBackend"]], Type["TrainerBackend"]]:
    """
    Register a training backend adapter.

    The decorator associates a normalized backend name with a
    :class:`TrainerBackend` subclass. Registered backends can subsequently
    be selected by name through :class:`BioTrainer`.

    Registering a backend also initializes its trainer registry, allowing
    trainer implementations to be registered with :func:`register_trainer`.

    Parameters
    ----------
    name
        Public name used to identify the backend. The name is stripped of
        leading and trailing whitespace and converted to lowercase.

    Returns
    -------
    Callable
        A class decorator that registers the backend class and returns it
        unchanged.

    Raises
    ------
    ValueError
        If ``name`` is empty or a backend with the same normalized name
        has already been registered.
    TypeError
        If the decorated object is not a subclass of
        :class:`TrainerBackend`.

    Examples
    --------
    ```python
    @register_backend("monai")
    class MonaiTrainerBackend(TrainerBackend):
        ...
    ```
    """
    backend_name = _validate_name(name, "Backend")

    def decorator(
        cls: Type["TrainerBackend"],
    ) -> Type["TrainerBackend"]:
        if not isinstance(cls, type) or not issubclass(cls, TrainerBackend):
            raise TypeError(
                "A backend must inherit from TrainerBackend."
            )

        if backend_name in BACKEND_REGISTRY:
            raise ValueError(
                f"Backend {backend_name!r} is already registered."
            )

        BACKEND_REGISTRY[backend_name] = cls
        TRAINER_REGISTRY.setdefault(backend_name, {})

        return cls

    return decorator

In [ ]:
#| export

def register_trainer(
    backend: str,
    name: str = "supervised",
) -> Callable[[TrainerFactory], TrainerFactory]:
    """
    Register a trainer implementation for a training backend.

    The decorator associates a normalized trainer name with a callable
    trainer implementation within a registered backend. The callable may
    be a trainer class, factory function, or other callable object.

    Trainer names are scoped to their backend, so the same name can be
    registered independently for different backends. For example,
    ``"supervised"`` may refer to a fastai trainer for the ``"fastai"``
    backend and a MONAI trainer for the ``"monai"`` backend.

    Parameters
    ----------
    backend
        Name of the registered backend to which the trainer belongs.
        The name is stripped of leading and trailing whitespace and
        converted to lowercase.
    name
        Public name used to identify the trainer within the backend.
        The name is normalized in the same way as ``backend``.

    Returns
    -------
    Callable
        A decorator that registers the trainer implementation and
        returns it unchanged.

    Raises
    ------
    ValueError
        If ``backend`` or ``name`` is empty, if ``backend`` has not been
        registered, or if a trainer with the same normalized name is
        already registered for the backend.
    TypeError
        If the decorated object is not callable.

    Notes
    -----
    ``register_trainer`` does not impose a specific trainer interface.
    The registered implementation is resolved by the backend and is
    expected to provide the functionality required by :class:`BioTrainer`.

    Examples
    --------
    Register a trainer class:

    ```python
    @register_trainer("fastai", "supervised")
    class FastaiTrainer:
        ...
    ```

    Register a trainer factory:

    ```python
    @register_trainer("monai", "supervised")
    def create_supervised(*args, **kwargs):
        return SupervisedTrainer(*args, **kwargs)
    ```
    """
    backend_name = _validate_name(backend, "Backend")
    trainer_name = _validate_name(name, "Trainer")

    if backend_name not in BACKEND_REGISTRY:
        raise ValueError(
            f"Cannot register trainer {trainer_name!r}: "
            f"backend {backend_name!r} is not registered."
        )

    def decorator(factory: TrainerFactory) -> TrainerFactory:
        if not callable(factory):
            raise TypeError(
                "A trainer must be a callable class or factory."
            )

        trainers = TRAINER_REGISTRY[backend_name]

        if trainer_name in trainers:
            raise ValueError(
                f"Trainer {trainer_name!r} is already registered "
                f"for backend {backend_name!r}."
            )

        trainers[trainer_name] = factory
        return factory

    return decorator

### Configuration

In [ ]:
#| export

@dataclass
class TrainerConfig:
    """
    Backend-independent configuration for :class:`BioTrainer`.

    This class defines the canonical vocabulary used by bioMONAI. Backend
    adapters translate these values into the corresponding native training
    framework API.

    Parameters
    ----------
    model
        Model or network to train.
    dls
        bioMONAI DataLoaders containing training and validation data.
    optimizer
        Optimizer instance or optimizer factory.
    loss
        Loss function used during training.
    metrics
        Metrics evaluated during validation.
    epochs
        Number of training epochs.
    lr
        Learning rate, when applicable.
    validate
        Whether validation should be performed during training.
    valid_every
        Number of epochs between validation runs. ``1`` means validation
        after every epoch.
    device
        Training and evaluation device.
    callbacks
        Backend-independent callbacks or lifecycle handlers.
    inferer
        Optional inference strategy used during validation.
    postprocessing
        Optional post-processing applied during validation.
    backend
        Training backend identifier.
    backend_kwargs
        Backend-specific arguments that do not belong in the common API.

    Notes
    -----
    The names in this class are deliberately independent of native
    frameworks. For example, bioMONAI uses ``epochs`` rather than MONAI's
    ``max_epochs`` and ``loss`` rather than ``loss_function``.
    """

    model: Any = None
    dls: Any = None

    optimizer: Any = None
    loss: Any = None
    metrics: Any = None

    epochs: int = 1
    lr: Optional[float] = None

    validate: bool = True
    valid_every: int = 1

    device: Any = None
    callbacks: Any = None

    inferer: Any = None
    postprocessing: Any = None

    csv_logger: bool = False
    show_graph: bool = False
    show_results: bool = False
    find_lr: bool = False
    find_lr_kwargs: Dict[str, Any] = field(default_factory=dict)
    save_dir: str | Path = 'models'

    backend: str = "monai"
    trainer: str = "supervised"

    backend_kwargs: Dict[str, Any] = field(default_factory=dict)

In [ ]:
#| export

def _validate_trainer_config(config: TrainerConfig):
    """
    Validate backend-independent trainer configuration.

    Parameters
    ----------
    config
        Trainer configuration to validate.

    Raises
    ------
    ValueError
        If an invalid epoch count, validation frequency, or backend name
        is supplied.
    """
    if config.epochs < 1:
        raise ValueError("epochs must be >= 1")

    if config.valid_every < 1:
        raise ValueError("valid_every must be >= 1")

    config.backend = _validate_name(
        config.backend,
        "Backend",
    )

In [ ]:
#| export

def _native_signature(
    self,
    target=None,
):
    """
    Return the signature of a native trainer or callable.

    Parameters
    ----------
    target
        Callable or class to inspect. If omitted, ``trainer_cls`` is used.

    Returns
    -------
    inspect.Signature or None
        Native signature, or ``None`` if it cannot be inspected.
    """
    target = target or self.trainer_cls

    if target is None:
        return None

    try:
        return inspect.signature(target)
    except (TypeError, ValueError):
        return None


In [ ]:
#| export

def _accepts_kwargs(
    self,
    target=None,
) -> bool:
    """
    Check whether a native callable accepts arbitrary keyword arguments.

    Parameters
    ----------
    target
        Callable or class to inspect.

    Returns
    -------
    bool
        ``True`` when the callable defines ``**kwargs``.
    """
    signature = self._native_signature(target)

    if signature is None:
        return False

    return any(
        parameter.kind == inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    )

In [ ]:
#| export

def _translate_kwargs(
    self,
    values: Mapping[str, Any],
    target=None,
    *,
    drop_none: bool = True,
) -> Dict[str, Any]:
    """
    Translate canonical arguments to native framework arguments.

    Parameters
    ----------
    values
        Mapping containing canonical bioMONAI argument names.
    target
        Native callable that will receive the translated arguments.
    drop_none
        If ``True``, arguments whose value is ``None`` are omitted.

    Returns
    -------
    dict
        Arguments accepted by the native callable.

    Notes
    -----
    Translation happens in two stages:

    1. canonical names are renamed using ``arg_map``;
    2. unsupported arguments are removed using the native signature.

    For example, a MONAI backend can translate:

    ``epochs`` -> ``max_epochs``

    ``model`` -> ``network``

    ``loss`` -> ``loss_function``
    """
    target = target or self.trainer_cls

    translated = {}

    for name, value in values.items():

        if drop_none and value is None:
            continue

        native_name = self.arg_map.get(name, name)

        translated[native_name] = value

    signature = self._native_signature(target)

    if signature is None or self._accepts_kwargs(target):
        return translated

    allowed = set(signature.parameters)

    return {
        name: value
        for name, value in translated.items()
        if name in allowed
    }

In [ ]:
#| export

def _common_kwargs(
    self,
) -> Dict[str, Any]:
    """
    Collect arguments shared by the canonical trainer API.

    Returns
    -------
    dict
        Canonical training arguments.

    Notes
    -----
    Validation scheduling options such as ``validate`` and ``valid_every``
    are intentionally excluded. They control orchestration rather than
    native trainer construction.
    """
    cfg = self.config

    return {
        "model": cfg.model,
        "dls": cfg.dls,
        "optimizer": cfg.optimizer,
        "loss": cfg.loss,
        "metrics": cfg.metrics,
        "epochs": cfg.epochs,
        "lr": cfg.lr,
        "device": cfg.device,
        "inferer": cfg.inferer,
        "postprocessing": cfg.postprocessing,
    }

In [ ]:
#| export

def _backend_kwargs(
    self,
    target=None,
) -> Dict[str, Any]:
    """
    Translate explicitly supplied backend-specific arguments.

    Parameters
    ----------
    target
        Native callable that will receive the arguments.

    Returns
    -------
    dict
        Backend-specific arguments accepted by the target.

    Notes
    -----
    ``backend_kwargs`` is an escape hatch for framework-specific features.
    Functionality shared across multiple backends should instead be exposed
    through the canonical ``BioTrainer`` API.
    """
    return self._translate_kwargs(
        self.config.backend_kwargs,
        target=target,
        drop_none=False,
    )

In [ ]:
#| export
def _get_trainer(self):
    """
    Return the trainer factory registered for the current backend and
    trainer type.
    """
    backend_trainers = TRAINER_REGISTRY.get(
        self.config.backend,
        {},
    )

    trainer = backend_trainers.get(
        self.config.trainer,
    )

    if trainer is None:
        available = ", ".join(
            sorted(backend_trainers)
        )

        raise ValueError(
            f"Unknown trainer {self.config.trainer!r} for "
            f"backend {self.config.backend!r}. "
            f"Available trainers: {available or 'none'}"
        )

    return trainer

### Trainer Backend

In [ ]:
#| export

class TrainerBackend:
    """
    Base class for bioMONAI training backends.

    A backend translates the canonical :class:`TrainerConfig` into the
    native API of a particular training framework.

    Backend implementations are responsible for:

    - translating common argument names;
    - adapting bioMONAI objects to native objects;
    - constructing the native trainer;
    - implementing validation;
    - implementing backend-specific training lifecycle logic.

    Parameters
    ----------
    config
        Canonical trainer configuration.

    Notes
    -----
    Framework-specific concepts should remain inside the backend. For
    example, MONAI/Ignite events should not become part of the public
    ``BioTrainer`` API.
    """

    arg_map: Mapping[str, str] = {}
    trainer_cls: Any = None

    def __init__(self, config: TrainerConfig):
        self.config = config
        self.trainer = None
        self.evaluator = None


TrainerBackend._native_signature = _native_signature
TrainerBackend._accepts_kwargs = _accepts_kwargs
TrainerBackend._translate_kwargs = _translate_kwargs
TrainerBackend._common_kwargs = _common_kwargs
TrainerBackend._backend_kwargs = _backend_kwargs
TrainerBackend._get_trainer = _get_trainer

In [ ]:
#| export

def _trainer_backend_fit(self):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement fit()."
    )


def _trainer_backend_validate(self):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement validate()."
    )


def _trainer_backend_predict(self, *args, **kwargs):
    raise NotImplementedError(
        f"{self.__class__.__name__} does not implement predict()."
    )


TrainerBackend.fit = _trainer_backend_fit
TrainerBackend.validate = _trainer_backend_validate
TrainerBackend.predict = _trainer_backend_predict

### BioTrainer

In [ ]:
#| export

class BioTrainer:
    """
    Backend-independent training interface for bioMONAI.

    ``BioTrainer`` exposes one canonical set of training arguments and
    delegates framework-specific implementation to a registered
    :class:`TrainerBackend`.

    The ``backend`` selects the underlying training framework, while
    ``trainer`` selects the training strategy implemented by that backend.
    For example, ``backend="fastai", trainer="supervised"`` uses the
    bioMONAI ``fastTrainer`` implementation, while another trainer type
    could provide GAN-specific training.

    The same user-facing arguments can therefore be used with MONAI,
    fastai, PyTorch, Keras, Ignite, or other supported backends.

    Parameters
    ----------
    model
        Model or network to train.
    dls
        bioMONAI DataLoaders containing training and validation data.
    optimizer
        Optimizer instance or factory.
    loss
        Loss function.
    metrics
        Validation metrics.
    epochs
        Number of training epochs.
    lr
        Learning rate.
    validate
        Whether to perform validation during training.
    valid_every
        Number of epochs between validation runs.
    device
        Training device.
    callbacks
        Backend-independent callbacks.
    inferer
        Optional inference strategy.
    postprocessing
        Optional validation post-processing.
    csv_logger
        Whether to enable CSV logging when supported by the selected
        trainer.
    show_graph
        Whether to display training graphs when supported by the selected
        trainer.
    show_results
        Whether to display training results when supported by the selected
        trainer.
    find_lr
        Whether to perform learning-rate finding when supported by the
        selected trainer.
    find_lr_kwargs
        Additional arguments passed to the learning-rate finder.
    save_dir
        Directory used for saved models and training artifacts.
    backend
        Training backend identifier, such as ``"fastai"`` or ``"monai"``.
    trainer
        Training strategy registered for the selected backend, such as
        ``"supervised"`` or ``"gan"``.
    backend_kwargs
        Optional backend-specific arguments that are not part of the
        common ``BioTrainer`` interface.

    Notes
    -----
    ``BioTrainer`` deliberately does not expose native framework names such
    as MONAI's ``max_epochs`` or ``loss_function``. Backend adapters perform
    that translation.

    ``backend`` and ``trainer`` are independent. A backend can provide
    multiple trainer implementations, and the same trainer name can be
    implemented by multiple backends.
    """

    def __init__(
        self,
        model=None,
        dls=None,
        *,
        optimizer=None,
        loss=None,
        metrics=None,
        epochs=1,
        lr=None,
        validate=True,
        valid_every=1,
        device=None,
        callbacks=None,
        inferer=None,
        postprocessing=None,
        csv_logger=False,
        show_graph=False,
        show_results=False,
        find_lr=False,
        find_lr_kwargs=None,
        save_dir="models",
        backend="torch",
        trainer="supervised",
        backend_kwargs=None,
    ):
        self.config = TrainerConfig(
            model=model,
            dls=dls,
            optimizer=optimizer,
            loss=loss,
            metrics=metrics,
            epochs=epochs,
            lr=lr,
            validate=validate,
            valid_every=valid_every,
            device=device,
            callbacks=callbacks,
            inferer=inferer,
            postprocessing=postprocessing,
            csv_logger=csv_logger,
            show_graph=show_graph,
            show_results=show_results,
            find_lr=find_lr,
            find_lr_kwargs=find_lr_kwargs or {},
            save_dir=save_dir,
            backend=backend,
            trainer=trainer,
            backend_kwargs=backend_kwargs or {},
        )

        _validate_trainer_config(self.config)

        self.backend = self._build_backend()

    def _build_backend(self):
        """
        Instantiate the registered backend selected by ``config.backend``.

        Returns
        -------
        TrainerBackend
            Backend responsible for implementing the selected training
            strategy.

        Raises
        ------
        ValueError
            If the requested backend has not been registered.
        """
        backend_cls = BACKEND_REGISTRY.get(self.config.backend)

        if backend_cls is None:
            available = ", ".join(sorted(BACKEND_REGISTRY))

            raise ValueError(
                f"Unknown training backend "
                f"'{self.config.backend}'. "
                f"Available backends: {available}"
            )

        return backend_cls(self.config)

    def fit(self):
        """
        Train the model using the selected backend and trainer.

        Validation is automatically performed according to ``validate`` and
        ``valid_every`` when supported by the selected trainer.

        Returns
        -------
        BioTrainer
            The trainer instance.
        """
        self.backend.fit()
        return self

    def validate(self):
        """
        Run validation independently of training.

        Returns
        -------
        Any
            Backend-specific validation result.
        """
        return self.backend.validate()

    def predict(self, *args, **kwargs):
        """
        Run inference using the selected backend.

        Parameters
        ----------
        *args
            Positional backend-specific arguments.
        **kwargs
            Keyword backend-specific arguments.

        Returns
        -------
        Any
            Backend-specific prediction result.
        """
        return self.backend.predict(*args, **kwargs)

    @property
    def trainer_instance(self):
        """
        Return the instantiated trainer.

        Returns
        -------
        Any
            The native or bioMONAI trainer instance created by the selected
            backend and trainer implementation.

        Examples
        --------
        With ``backend="fastai", trainer="supervised"``, this returns the
        bioMONAI ``fastTrainer`` instance.

        With ``backend="monai", trainer="supervised"``, this may return a
        MONAI ``SupervisedTrainer`` instance.
        """
        return self.backend.trainer

    @property
    def evaluator_instance(self):
        """
        Return the instantiated validation evaluator, when applicable.

        Returns
        -------
        Any or None
            The backend evaluator, or ``None`` when validation is integrated
            directly into the trainer.
        """
        return self.backend.evaluator

## fastai Engines

In [ ]:
#| export
def fastai_build_trainer(self):
    """
    Build the trainer selected by ``config.trainer``.

    Returns
    -------
    FastaiTrainerBackend
        The backend instance with ``self.trainer`` initialized.
    """
    trainer_factory = self._get_trainer()

    kwargs = self._common_kwargs()
    kwargs.update(self.config.backend_kwargs)

    self.trainer = trainer_factory(**kwargs)

    return self

In [ ]:
#| export

def fastai_fit(self):
    """
    Train using the configured fastai trainer.

    Returns
    -------
    Any
        Result returned by the underlying trainer.
    """
    if self.trainer is None:
        self._build_trainer()

    return self.trainer.fit()


In [ ]:
#| export

def fastai_validate(self):
    """
    Run validation using the configured fastai trainer.

    Returns
    -------
    Any
        Validation result.
    """
    if self.trainer is None:
        self._build_trainer()

    if hasattr(self.trainer, "validate"):
        return self.trainer.validate()

    if self.evaluator is not None:
        return self.evaluator.validate()

    raise RuntimeError(
        f"Trainer {type(self.trainer).__name__!r} does not provide "
        "a validation interface."
    )


In [ ]:
#| export

def fastai_predict(self, *args, **kwargs):
    """
    Run inference using the configured fastai trainer.

    Parameters
    ----------
    *args
        Positional arguments passed to the trainer.
    **kwargs
        Keyword arguments passed to the trainer.

    Returns
    -------
    Any
        Prediction result returned by the trainer.
    """
    if self.trainer is None:
        self._build_trainer()

    if not hasattr(self.trainer, "predict"):
        raise RuntimeError(
            f"Trainer {type(self.trainer).__name__!r} does not provide "
            "a prediction interface."
        )

    return self.trainer.predict(*args, **kwargs)

In [ ]:
#| export
class FastaiTrainerBackend(TrainerBackend):
    """
    fastai training backend for bioMONAI.

    This backend adapts the backend-independent :class:`TrainerConfig`
    interface to bioMONAI's fastai-based training implementations.

    The trainer implementation is selected through ``config.trainer`` and
    resolved from ``TRAINER_REGISTRY["fastai"]``. For example::

        BioTrainer(
            model=model,
            dls=dls,
            backend="fastai",
            trainer="supervised",
        )

    resolves the ``"supervised"`` trainer registered for the fastai backend.

    Notes
    -----
    The registered trainer is expected to be a bioMONAI trainer such as
    ``fastTrainer`` rather than a native fastai ``Learner``.
    """

    arg_map = {
        "dls": "dataloaders",
        "loss": "loss_func",
        "save_dir": "model_dir",
    }


FastaiTrainerBackend._build_trainer = fastai_build_trainer
FastaiTrainerBackend.fit = fastai_fit
FastaiTrainerBackend.validate = fastai_validate
FastaiTrainerBackend.predict = fastai_predict

FastTrainer is used for training models in bioinformatics applications, where specific loss functions and optimizers oriented to biological data can be used. 

In [ ]:
#| export
class fastTrainer(Learner):
    """
    A custom implementation of the FastAI Learner class for training models in bioinformatics applications.

    """
    
    def __init__(self, 
                 dataloaders: DataLoaders = None, # The DataLoader objects containing training and validation datasets.
                 model: callable = None, # A callable model that will be trained on the dataset.
                 loss_fn: Any | None = None, # The loss function to optimize during training. If None, defaults to a suitable default.
                 optimizer: Optimizer | OptimWrapper = Adam, # The optimizer function to use. Defaults to Adam if not specified.
                 lr: float | slice = 1e-3, # Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
                 splitter: callable = trainable_params, # 
                 callbacks: Callback | MutableSequence | None = None, # A callable that determines which parameters of the model should be updated during training.
                 metrics: Any | MutableSequence | None = None, # Optional list of callback functions to customize training behavior.
                 csv_log: bool = False, # Metrics to evaluate the performance of the model during training.
                 show_graph: bool = True, # Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
                 show_graph: bool = True, # The base directory where models are saved or loaded from. Defaults to None.
                 find_lr: bool = False, # Subdirectory within the base path where trained models are stored. Default is 'models'.
                 find_lr_fn = valley, # Weight decay factor for optimization. Defaults to None.
                 path: str | Path | None = None, # Whether to apply weight decay to batch normalization and bias parameters.
                 model_dir: str | Path = 'models', # Whether to update the batch normalization statistics during training.
                 wd: float | int | None = None, 
                 wd_bn_bias: bool = False, 
                 train_bn: bool = True, 
                 moms: tuple = (0.95,0.85,0.95), # Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
                 default_cbs: bool = True, # Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
                 ):
        cbs = callbacks if callbacks is not None else []  # Ensure cbs is a list
        if default_cbs:
            if show_graph:
                cbs.append(ShowGraphCallback())
            if csv_log:
                cbs.append(CSVLogger(fname='history.csv', append=False))
        
        super().__init__(dataloaders, model, loss_fn, optimizer, lr, splitter, cbs, metrics, path, model_dir, wd, wd_bn_bias, train_bn, moms)
        
        if show_summary:
                print(self.summary())
        if find_lr:
                lr = self.lr_find(suggest_funcs=find_lr_fn)
                self.lr = float('%.1g'%(lr))
                print('Inferred learning rate: ', self.lr)
        

    @classmethod
    def from_yaml(cls, dataloaders, model, yaml_path):
        """
        Method to read from a YAML file and obtain the parameters for FastTrainer.
        """
        # Read the configuration from the yaml file and replace None strings with Nonetype values
        config = read_yaml(yaml_path)
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        
        
        # OBTAIN THE LOSS FUNCTION
        loss_str = config.get('loss_fn', None)
        # Look for the loss function within the variables and check if its valid
        if loss_str == None:
            loss_func = None
        else:
            lf_cls = globals().get(loss_str) or getattr(fm, loss_str, None)
            if lf_cls is None:
                raise ValueError(f"Loss function '{loss_str}' not found or invalid.")
            loss_func = lf_cls()


        # OBTAIN THE OPTIMIZER
        opt_str = config.get('optimizer', 'Adam')
        # Look for the optimizer within the variables and check if its valid
        if isinstance(opt_str, str):
            opt_cls = globals().get(opt_str, None)
            if opt_cls is None and hasattr(fastai.optimizer, opt_str):
                opt_cls = getattr(fastai.optimizer, opt_str)
            if opt_cls is None and hasattr(toptim, opt_str):
                opt_cls = getattr(toptim, opt_str)
            if opt_cls is None or not callable(opt_cls):
                raise ValueError(f"Optimizer '{opt_str}' not found or invalid.")
            opt_func = opt_cls


        # OBTAIN THE METRICS
        metrics_cfg = config.get('metrics', [])
        # Look for the metrics within the variables and check if they are valid
        valid_metrics = []
        if metrics_cfg == None:
            valid_metrics = None
        else:
            valid_metrics = dictlist_to_funclist(metrics_cfg)

        # OBTAIN THE CALLBACKS
        callbacks = []
        cbs = config.get('callbacks', [])
        callbacks = dictlist_to_funclist(cbs)

        # OBTAIN THE SPLITTER
        splitter_str = config.get('splitter', trainable_params)
        if isinstance(splitter_str, str):
            splitter_func = globals().get(splitter_str) or getattr(fastai.learner, splitter_str, None)
        elif splitter_str is None:
            splitter_func = trainable_params 
        else:
                splitter_func = splitter_str


        # OBTAIN OTHER PARAMETERS FROM THE YAML CONFIGURATION 
        lr = config.get('lr', 1e-3)
        csv_log = config.get('csv_log', False)
        show_graph = config.get('show_graph', True)
        show_summary = config.get('show_summary', False)
        path = config.get('path', None)
        model_dir = config.get('model_dir', 'models')
        wd = config.get('wd', None)
        wd_bn_bias = config.get('wd_bn_bias', False)
        train_bn = config.get('train_bn', True)
        moms = config.get('moms', (0.95,0.85,0.95))
   


        # Return all the parameters
        return cls(dataloaders = dataloaders, model = model, loss_fn = loss_func, optimizer = opt_func,
            lr = lr, splitter = splitter_func, callbacks = callbacks, metrics = valid_metrics, path = path,
            model_dir = model_dir, wd = wd, wd_bn_bias = wd_bn_bias, train_bn = train_bn, moms = moms,
            csv_log = csv_log, show_graph = show_graph, show_summary = show_summary          
        )        

In [ ]:
#| export
@register_trainer("fastai", "supervised")
def _fastai_supervised(
    *args: Any,
    **kwargs: Any,
) -> Any:
    """
    Construct the fastai-based bioMONAI supervised trainer.

    This trainer provides the standard supervised training workflow
    for the fastai backend and delegates training behavior to
    :class:`fastTrainer`.

    Parameters
    ----------
    *args
        Positional arguments forwarded to ``fastTrainer``.
    **kwargs
        Keyword arguments forwarded to ``fastTrainer``.

    Returns
    -------
    fastTrainer
        Configured fastai-based training engine.
    """
    return fastTrainer(*args, **kwargs)

#### Example: train a model with configuration from a YAML file.

In [ ]:
from monai.networks.nets import SEResNet50
from bioMONAI.data import BioDataLoaders
from bioMONAI.metrics import BalancedAccuracy, Precision, accuracy

In [ ]:
# Import the data
image_path = '_data'
info = download_medmnist('bloodmnist', image_path, download_only=True)
batch_size = 32
path = Path(image_path)/'bloodmnist'
path_train = path/'train'
path_val = path/'val'

Dataset 'bloodmnist' is already downloaded and available in '_data/bloodmnist'.


In [ ]:
# Define the dataloader
data = BioDataLoaders.class_from_folder(
    path,
    train='train',
    valid='val',
    vocab=info['label'],
    batch_tfms=None,
    bs=batch_size)

# Define the model
model = SEResNet50(spatial_dims=2,
                   in_channels=3,   
                   num_classes=8)    

In [ ]:
# Define the trainer with configuration from a YAML file 
# yaml_path = "./data_examples/sample_config.yml"
# trainer = fastTrainer.from_yaml(data, model, yaml_path)

# # Train the model
# trainer.fit(1)


In [ ]:
# print(trainer.recorder.metric_names)

In [ ]:
#| export
def _add_norm(dls, meta, pretrained, n_in=3):
    if not pretrained: return
    stats = meta.get('stats')
    if stats is None: return
    if n_in != len(stats[0]): return
    if not dls.after_batch.fs.filter(risinstance(Normalize)):
        dls.add_tfms([Normalize.from_stats(*stats)],'after_batch')

def _timm_norm(dls, cfg, pretrained, n_in=3):
    if not pretrained: return
    if n_in != len(cfg['mean']): return
    if not dls.after_batch.fs.filter(risinstance(Normalize)):
        tfm = Normalize.from_stats(cfg['mean'],cfg['std'])
        dls.add_tfms([tfm],'after_batch')

VisionTrainer is used for computer vision applications, where image normalization or other computer vision related settings are needed.

In [ ]:
#| export
@delegates(create_vision_model)
def visionTrainer(  dataloaders: DataLoaders, # The DataLoader objects containing training and validation datasets.
                    model: callable, # A callable model that will be trained on the dataset.
                    normalize=True, 
                    n_out=None, 
                    pretrained=True, 
                    weights=None,
                    # Trainer args
                    loss_fn: Any | None = None, # The loss function to optimize during training. If None, defaults to a suitable default.
                    optimizer: Optimizer | OptimWrapper = Adam, # The optimizer function to use. Defaults to Adam if not specified.
                    lr: float | slice = 1e-3, # Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
                    splitter: callable = trainable_params, # 
                    callbacks: Callback | MutableSequence | None = None, # A callable that determines which parameters of the model should be updated during training.
                    metrics: Any | MutableSequence | None = None, # Optional list of callback functions to customize training behavior.
                    csv_log: bool = False, # Metrics to evaluate the performance of the model during training.
                    show_graph: bool = True, # Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
                    show_summary: bool = False, # The base directory where models are saved or loaded from. Defaults to None.
                    find_lr: bool = False, # Subdirectory within the base path where trained models are stored. Default is 'models'.
                    find_lr_fn = valley, # Weight decay factor for optimization. Defaults to None.
                    path: str | Path | None = None, # Whether to apply weight decay to batch normalization and bias parameters.
                    model_dir: str | Path = 'models', # Whether to update the batch normalization statistics during training.
                    wd: float | int | None = None, 
                    wd_bn_bias: bool = False, 
                    train_bn: bool = True, 
                    moms: tuple = (0.95,0.85,0.95), # Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
                    default_cbs: bool = True, # Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
                    # model & head args
                    cut=None, 
                    init=kaiming_normal_, 
                    custom_head=None, 
                    concat_pool=True, 
                    pool=True,
                    lin_ftrs=None, 
                    ps=0.5, 
                    first_bn=True, 
                    bn_final=False, 
                    lin_first=False, 
                    y_range=None, 
                    **kwargs):
    "Build a vision trainer from `dataloaders` and `model`"
    if n_out is None: n_out = get_c(dataloaders)
    assert n_out, "`n_out` is not defined, and could not be inferred from data, set `dataloaders.c` or pass `n_out`"
    meta = model_meta.get(model, {'cut':cut, 'split':default_split})
    model_args = dict(init=init, custom_head=custom_head, concat_pool=concat_pool, pool=pool, lin_ftrs=lin_ftrs, ps=ps,
                      first_bn=first_bn, bn_final=bn_final, lin_first=lin_first, y_range=y_range, **kwargs)
    n_in = kwargs['n_in'] if 'n_in' in kwargs else 3
    if isinstance(model, str):
        model,cfg = create_timm_model(model, n_out, default_split, pretrained, **model_args)
        if normalize: _timm_norm(dataloaders, cfg, pretrained, n_in)
    else:
        if normalize: _add_norm(dataloaders, meta, pretrained, n_in)
        model = create_vision_model(model, n_out, pretrained=pretrained, weights=weights, **model_args)

    splitter = ifnone(splitter, meta['split'])
    trainer = fastTrainer(dataloaders, model, loss_fn=loss_fn, optimizer=optimizer, lr=lr, splitter=splitter, callbacks=callbacks, csv_log=csv_log, 
                        show_graph=show_graph, show_summary=show_summary, find_lr=find_lr, find_lr_fn=find_lr_fn, metrics=metrics, path=path, 
                        model_dir=model_dir, wd=wd, wd_bn_bias=wd_bn_bias, train_bn=train_bn, moms=moms, default_cbs=default_cbs)
    if pretrained: trainer.freeze()
    # keep track of args for loggers
    store_attr('model,normalize,n_out,pretrained', self=trainer, **kwargs)
    return trainer

In [ ]:
#| export
@register_trainer("fastai", "vision")
def _fastai_vision(
    *args: Any,
    **kwargs: Any,
) -> Any:
    """
    Construct the fastai-based vision trainer.

    This trainer provides a vision-specific training workflow using
    :class:`visionTrainer`.

    Parameters
    ----------
    *args
        Positional arguments forwarded to ``visionTrainer``.
    **kwargs
        Keyword arguments forwarded to ``visionTrainer``.

    Returns
    -------
    visionTrainer
        Configured vision training engine.
    """
    return visionTrainer(*args, **kwargs)

## Monai Engines

In [ ]:
#| export

@register_backend("monai")
class MonaiTrainerBackend(TrainerBackend):
    """
    MONAI implementation of the bioMONAI trainer backend.

    The backend composes MONAI's ``SupervisedTrainer`` and
    ``SupervisedEvaluator`` to provide the validation semantics defined by
    ``BioTrainer``.

    The resulting lifecycle is:

        train epoch
            ↓
        validation
            ↓
        train epoch
            ↓
        validation

    Validation frequency is controlled by ``valid_every``.

    This composition is intentional: MONAI's native trainer and evaluator
    remain separate objects, while ``BioTrainer`` presents them as one
    coherent training interface.
    """

    arg_map = {
        "device": "device",
        "epochs": "max_epochs",
        "optimizer": "optimizer",
        "model": "network",
        "loss": "loss_function",
        "inferer": "inferer",
        "postprocessing": "postprocessing",
    }

    def __init__(self, config: TrainerConfig):
        super().__init__(config)

        from monai.engines import (
            SupervisedEvaluator,
            SupervisedTrainer,
        )

        self.trainer_cls = SupervisedTrainer
        self.evaluator_cls = SupervisedEvaluator

    @property
    def train_loader(self):
        """Return the training DataLoader."""
        return self.config.dls.train

    @property
    def valid_loader(self):
        """
        Return the validation DataLoader from the bioMONAI DataLoaders object.

        Returns
        -------
        Any or None
            Validation DataLoader, or ``None`` when no validation split exists.
        """
        if self.config.dls is None:
            return None

        return getattr(self.config.dls, "valid", None)

    def _build_trainer(self):
        """
        Construct the native MONAI ``SupervisedTrainer``.

        The canonical bioMONAI configuration is translated into MONAI's
        terminology and expected objects before construction.

        Examples of argument translation include:

        - ``epochs`` → ``max_epochs``
        - ``model`` → ``network``
        - ``loss`` → ``loss_function``
        - ``dls.train`` → ``train_data_loader``
        """
        from monai.engines import SupervisedTrainer

        values = self._common_kwargs()

        # ``dls`` is a bioMONAI DataLoaders object, whereas MONAI expects
        # the actual training DataLoader.
        values["train_data_loader"] = self.train_loader
        values.pop("dls", None)

        kwargs = self._translate_kwargs(
            values,
            target=SupervisedTrainer,
        )

        kwargs.update(
            self._backend_kwargs(
                target=SupervisedTrainer,
            )
        )

        self.trainer = SupervisedTrainer(**kwargs)

        return self.trainer

    def _build_evaluator(self):
        """
        Construct the native MONAI ``SupervisedEvaluator``.

        The evaluator is created only when validation is enabled. It receives
        the validation DataLoader and the model being trained.

        Returns
        -------
        SupervisedEvaluator or None
            Constructed evaluator, or ``None`` when validation is disabled.

        Raises
        ------
        ValueError
            If validation is enabled but no validation DataLoader is available.
        """
        from monai.engines import SupervisedEvaluator

        cfg = self.config

        if not cfg.validate:
            return None

        if self.valid_loader is None:
            raise ValueError(
                "Validation is enabled, but no validation DataLoader "
                "is available."
            )

        values = {
            "device": cfg.device,
            "val_data_loader": self.valid_loader,
            "network": cfg.model,
            "inferer": cfg.inferer,
            "postprocessing": cfg.postprocessing,
        }

        kwargs = self._translate_kwargs(
            values,
            target=SupervisedEvaluator,
        )

        kwargs.update(
            self._backend_kwargs(
                target=SupervisedEvaluator,
            )
        )

        self.evaluator = SupervisedEvaluator(**kwargs)

        return self.evaluator

    def _attach_validation(self):
        """
        Attach scheduled validation to the MONAI training engine.

        Validation is triggered by Ignite's ``EPOCH_COMPLETED`` event and is
        executed every ``valid_every`` epochs.

        Notes
        -----
        This method is the bridge between the backend-independent BioTrainer
        contract and MONAI/Ignite's event-driven execution model.
        """
        if not self.config.validate:
            return

        from ignite.engine import Events

        evaluator = self.evaluator
        valid_every = self.config.valid_every

        @self.trainer.on(Events.EPOCH_COMPLETED)
        def _run_validation(engine):
            if engine.state.epoch % valid_every == 0:
                evaluator.run()

    def fit(self):
        """
        Train the model using MONAI.

        When validation is enabled, a ``SupervisedEvaluator`` is automatically
        executed according to ``valid_every``.

        Returns
        -------
        MonaiTrainerBackend
            This backend instance.
        """
        self._build_trainer()

        if self.config.validate:
            self._build_evaluator()
            self._attach_validation()

        self.trainer.run()

        return self

    def validate(self):
        """
        Run MONAI validation independently of training.

        Returns
        -------
        ignite.engine.State
            Evaluator state containing the validation results and metrics.

        Raises
        ------
        ValueError
            If validation is disabled.
        """
        if not self.config.validate:
            raise ValueError(
                "Validation is disabled for this trainer."
            )

        if self.evaluator is None:
            self._build_evaluator()

        self.evaluator.run()

        return self.evaluator.state

In [ ]:
#| export

def _monai_trainer(
    class_name: str,
    *args: Any,
    **kwargs: Any,
) -> Any:
    """
    Construct a trainer engine from ``monai.engines``.

    Parameters
    ----------
    class_name : str
        Name of the trainer class exposed by ``monai.engines``.
    *args
        Positional arguments forwarded to the trainer constructor.
    **kwargs
        Keyword arguments forwarded to the trainer constructor.

    Returns
    -------
    Any
        Instantiated MONAI training engine.

    Raises
    ------
    ImportError
        If MONAI is not installed or the requested engine is not
        available in the installed MONAI version.
    """
    try:
        from monai import engines
        trainer_class = getattr(engines, class_name)
    except (ImportError, AttributeError) as exc:
        raise ImportError(
            "MONAI trainer {!r} is unavailable; install MONAI with "
            "engine support or register a replacement trainer.".format(
                class_name
            )
        ) from exc

    return trainer_class(*args, **kwargs)


@register_trainer("monai", "supervised")
def _monai_supervised(
    *args: Any,
    **kwargs: Any,
) -> Any:
    """
    Construct a MONAI ``SupervisedTrainer``.

    The trainer implements MONAI's standard supervised training loop,
    including model optimization, loss computation, and the associated
    engine events.

    Parameters
    ----------
    *args
        Positional arguments forwarded to ``SupervisedTrainer``.
    **kwargs
        Keyword arguments forwarded to ``SupervisedTrainer``.

    Returns
    -------
    monai.engines.SupervisedTrainer
        Configured MONAI supervised training engine.
    """
    return _monai_trainer(
        "SupervisedTrainer",
        *args,
        **kwargs,
    )


@register_trainer("monai", "gan")
def _monai_gan(
    *args: Any,
    **kwargs: Any,
) -> Any:
    """
    Construct a MONAI ``GanTrainer``.

    This trainer provides MONAI's GAN-oriented training engine with
    support for generator/discriminator optimization and the associated
    engine workflow.

    Parameters
    ----------
    *args
        Positional arguments forwarded to ``GanTrainer``.
    **kwargs
        Keyword arguments forwarded to ``GanTrainer``.

    Returns
    -------
    monai.engines.GanTrainer
        Configured MONAI GAN training engine.
    """
    return _monai_trainer(
        "GanTrainer",
        *args,
        **kwargs,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()